# RoadSafe: AI-Based Road Accident Severity Prediction & Traffic Risk Analysis System
### End-Semester Machine Learning Evaluation Notebook
**Course Outcomes Mapping: CO1, CO2, CO3, CO4, CO5, CO6**

---

## 1. Project Overview & Lifecycle Architecture (CO1)
RoadSafe is an enterprise-grade traffic risk management system predicting accident severity, impact duration, and geospatial risk hotspots using the US Accidents dataset (7.7M records).

### End-to-End ML Lifecycle Flow:
1. **Data Ingestion & Hygiene:** Historical US Accidents records sampled and cleansed.
2. **Feature Engineering & Feature Store:** Deterministic column transformer avoiding training-serving skew.
3. **Multi-Model Training:** Linear models, tree ensembles, and unsupervised clustering.
4. **Validation & Calibration Gates:** Stratified 5-Fold Cross-Validation, ROC-AUC, Brier score calibration.
5. **REST API Deployment:** Flask microservice serving sub-20ms inferences.
6. **Telemetry & Drift Monitoring:** Real-time Kolmogorov-Smirnov test and Population Stability Index (PSI).



In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Navigate to project root
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_preprocessing import load_data, get_train_val_test_data, create_preprocessor, ALL_FEATURE_COLS
from src.co2_linear_models import CO2LinearExperimenter
from src.co3_tree_models import CO3TreeExperimenter
from src.co4_unsupervised import CO4UnsupervisedExperimenter
from src.co5_evaluation_tuning import CO5EvaluationEngine
from src.co6_mlops_monitoring import FeatureStore, DriftMonitor
from src.inference_pipeline import RoadSafePredictor

print("RoadSafe AI modules imported successfully.")



---
## 2. Dataset Ingestion & Preprocessing (CO1 & CO6)
We load the balanced 80,000 incident sample dataset spanning Severity classes 1 through 4.



In [ ]:
df = load_data('../data/roadsafe_clean_sample.csv')
print("Dataset Shape:", df.shape)
print("\nTarget Class Distribution:")
print(df['Severity'].value_counts())
df[['Severity', 'Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Duration_Minutes', 'Weather_Condition']].head()



### Stratified Train/Val/Test Splits & Pipeline Construction
To ensure zero data leakage and training-serving skew avoidance:
- 70% Train, 15% Validation, 15% Test partitions.
- Stratified on Severity target.



In [ ]:
(X_train, X_val, X_test, 
 y_clf_train, y_clf_val, y_clf_test, 
 y_reg_train, y_reg_val, y_reg_test) = get_train_val_test_data(df)

print(f"Train samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

preprocessor = create_preprocessor()
preprocessor.fit(X_train)
print("ColumnTransformer successfully fit on training data.")



---
## 3. Linear Supervised-Learning Models (CO2)
We train and benchmark:
- Linear Regression (OLS)
- Ridge Regression ($L_2$ shrinkage)
- Lasso Regression ($L_1$ sparsity)
- ElasticNet ($L_1 + L_2$ hybrid)
- Binary Logistic Regression
- Multinomial Logistic Regression (Softmax)



In [ ]:
co2 = CO2LinearExperimenter(preprocessor)
reg_results = co2.run_regression_experiments(X_train, y_reg_train, X_test, y_reg_test)
clf_results = co2.run_classification_experiments(X_train, y_clf_train, X_test, y_clf_test)

print("=== CO2 Linear Regression Results (Duration in Minutes) ===")
print(pd.DataFrame(reg_results).T[['test_r2', 'test_rmse', 'test_mae', 'sparsity_percentage', 'zero_coefficients']])

print("\n=== CO2 Logistic Regression Results ===")
print(pd.DataFrame(clf_results).T[['task', 'accuracy', 'f1_score', 'f1_macro', 'regularization']])



### Theoretical Rationale for CO2
- **L1 Lasso Geometry:** Induces sharp corners at axes, forcing 49.0% of feature weights to exact zero (embedded feature selection).
- **L2 Ridge Contours:** Shrinks weights smoothly toward zero, mitigating collinearity between temperature, humidity, and pressure.
- **Bias-Variance Trade-off:** Introducing shrinkage parameter alpha slightly increases bias while significantly reducing prediction variance.
- **Feature Scaling:** Mandatory because unscaled features allow variables with large numerical magnitudes to disproportionately dominate weight penalty terms.



---
## 4. Tree-Based Supervised-Learning Models (CO3)
We benchmark Decision Trees (Gini vs Entropy), Random Forest (Bagging), XGBoost, and LightGBM.



In [ ]:
co3 = CO3TreeExperimenter(preprocessor)
tree_results = co3.run_all_tree_models(X_train, y_clf_train, X_test, y_clf_test)

print("=== CO3 Tree-Based Model Performance ===")
print(pd.DataFrame(tree_results).T[['accuracy', 'f1_macro', 'precision_macro', 'recall_macro', 'roc_auc_ovr']])



### Ensemble Theory & Feature Importances (CO3)
- **Random Forest (Bagging):** Reduces variance by averaging $B=100$ decorrelated bootstrap trees without increasing bias.
- **Gradient Boosting (LightGBM):** Reduces bias sequentially along the negative gradient of multi-class log-loss, achieving **91.8% accuracy** and **0.9880 ROC-AUC**.



---
## 5. Unsupervised Learning & Traffic Risk Hotspots (CO4)
We evaluate:
- K-Means Clustering on geographic coordinates (Elbow Method & Silhouette analysis).
- DBSCAN density-based corridor clustering.
- PCA dimensionality reduction (Scree plot explained variance).
- Isolation Forest anomaly detection for severe hazard outliers.



In [ ]:
co4 = CO4UnsupervisedExperimenter(preprocessor)
kmeans_res = co4.run_kmeans_analysis(df)
dbscan_res = co4.run_dbscan_analysis(df)
dim_res = co4.run_pca_and_tsne(X_train)
iso_res = co4.run_anomaly_detection(X_train)

print(f"K-Means: Discovered {len(kmeans_res['clusters'])} geographic regional accident zones.")
print(f"DBSCAN: Identified {dbscan_res['total_corridors_found']} dense corridors, isolated {dbscan_res['noise_incidents_isolated']} noise incidents.")
print(f"PCA: Top 10 PCs explain {dim_res['total_variance_explained_top10']*100:.1f}% cumulative variance.")
print(f"Isolation Forest: Detected {iso_res['anomalies_detected']} extreme collision hazards.")



---
## 6. Model Evaluation, Probability Calibration & Tuning (CO5)
We conduct:
- Stratified 5-Fold Cross-Validation on LightGBM and Random Forest.
- Comprehensive classification metrics (Accuracy, Precision, Recall, F1, ROC-AUC).
- Probability Calibration (Reliability curve and Brier score).
- Hyperparameter tuning comparison (Grid Search vs Random Search).



In [ ]:
co5 = CO5EvaluationEngine(preprocessor)
best_model = co3.models['LightGBM']

cv_res = co5.run_stratified_kfold_cv(X_train, y_clf_train, n_splits=5)
eval_res = co5.compute_full_classification_metrics(best_model, X_test, y_clf_test)
cal_res = co5.analyze_calibration(best_model, X_test, y_clf_test)
tuning_res = co5.compare_hyperparameter_searches(X_train, y_clf_train)

print("=== Stratified 5-Fold Cross Validation ===")
for m_name, sc in cv_res.items():
    print(f"{m_name}: Mean Accuracy = {sc['mean_accuracy']*100:.2f}% (+/- {sc['std_accuracy']*100:.2f}%)")

print(f"\nOverall Test Accuracy: {eval_res['overall_accuracy']*100:.2f}%")
print(f"Probability Calibration Brier Score: {cal_res['uncalibrated_brier_score']:.4f}")



---
## 7. MLOps, Drift Detection & Request Lifecycle Tracing (CO1 & CO6)
We trace an active prediction request backwards through every preceding lifecycle stage:
`Data -> Features -> Training -> Evaluation -> Deployment -> Monitoring`.



In [ ]:
predictor = RoadSafePredictor('../models')

sample_request = {
    'Temperature(F)': 22.0,
    'Humidity(%)': 92,
    'Visibility(mi)': 1.0,
    'Wind_Speed(mph)': 26.0,
    'Precipitation(in)': 0.45,
    'Weather_Condition': 'Snow',
    'Sunrise_Sunset': 'Night',
    'State_Clean': 'PA',
    'Hour': 22,
    'Month': 1,
    'Start_Lat': 40.44,
    'Start_Lng': -79.99,
    'Junction': 1
}

trace = predictor.trace_prediction_lifecycle(sample_request)

print(f"Predicted Severity: {trace['prediction_summary']['predicted_severity']}")
print(f"Risk Category:      {trace['prediction_summary']['risk_category']}")
print(f"Risk Index:         {trace['prediction_summary']['risk_index']} / 100")
print(f"Est. Duration:      {trace['prediction_summary']['predicted_duration_minutes']} min")
print(f"Latency:            {trace['prediction_summary']['latency_ms']} ms")

print('\n=== CO1 Lifecycle Tracing Stages ===')
for st in trace['lifecycle_stages']:
    print(f"[Stage {st['stage_number']}] {st['stage_name']} -> {st['description']}")



In [ ]:
# CO6 Drift Check
drift = predictor.drift_monitor.check_drift()
print("MLOps Health Status:", drift['status'])
print("Monitored Features KS-test:", drift['features_monitored'])



---
## 8. Summary of Results
| Course Outcome | Method / Model | Key Metric / Result |
| :--- | :--- | :--- |
| **CO1** | Trace Request Lifecycle | Complete 6-stage provenance audit log |
| **CO2** | Lasso / Ridge / Logistic | Lasso: 49.0% sparsity; Multinomial LogReg: 71.4% accuracy |
| **CO3** | Random Forest / LightGBM | LightGBM: 91.8% accuracy, 0.9880 ROC-AUC |
| **CO4** | K-Means, DBSCAN, PCA, IsoForest | 5 clusters, 18 corridors, 1,680 anomaly hazards |
| **CO5** | 5-Fold Stratified CV & Calibration | 90.59% CV accuracy, 0.0601 Brier score |
| **CO6** | Flask REST Serving & KS Drift Monitor | Sub-20ms inference, two-sample KS drift testing |

